In [ ]:
import torch
import tiktoken
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from tqdm import tqdm
from collections import Counter

In [109]:
from modules.sampling import *
from modules.structure import *
from modules.training import *

In [31]:
import urllib.request
url = (
"https://raw.githubusercontent.com/rasbt/"
"LLMs-from-scratch/main/ch05/"
"01_main-chapter-code/gpt_download.py"
)
filename = url.split('/')[-1]
urllib.request.urlretrieve(url, filename)

('gpt_download.py', <http.client.HTTPMessage at 0x7d12dc6223c0>)

In [92]:
BASE_CONFIG = {
    "vocab_size": 50257, # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 1024, # Embedding dimension
    "n_heads": 16, # Number of attention heads
    "n_layers": 24, # Number of layers
    "drop_rate": 0.05, # Dropout rate
    "qkv_bias": True # Query-Key-Value bias
}

model_configs = {
    "gpt2-small (124M)": {"emb_dim": 768, "n_layers": 12, "n_heads": 12},
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16},
    "gpt2-large (774M)": {"emb_dim": 1280, "n_layers": 36, "n_heads": 20},
    "gpt2-xl (1558M)": {"emb_dim": 1600, "n_layers": 48, "n_heads": 25},
}

CHOOSE_MODEL = "gpt2-medium (355M)"
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])
model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")

In [88]:
from gpt_download import download_and_load_gpt2

settings, params = download_and_load_gpt2(
model_size=model_size,
models_dir="gpt2"
)

File already exists and is up-to-date: gpt2/355M/checkpoint
File already exists and is up-to-date: gpt2/355M/encoder.json
File already exists and is up-to-date: gpt2/355M/hparams.json
File already exists and is up-to-date: gpt2/355M/model.ckpt.data-00000-of-00001
File already exists and is up-to-date: gpt2/355M/model.ckpt.index
File already exists and is up-to-date: gpt2/355M/model.ckpt.meta
File already exists and is up-to-date: gpt2/355M/vocab.bpe


In [35]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [9]:
tokenizer = tiktoken.get_encoding("gpt2")

In [36]:
semicolon_id = tokenizer.encode(";")[0]

In [94]:
model = GPTModel(BASE_CONFIG)
model.eval()

load_weights_into_gpt(model, params)
model.to(device)

torch.manual_seed(123)
token_ids = generate(
model=model,
idx=text_to_token_ids("Every effort moves you", tokenizer).to(device),
max_new_tokens=25,
context_size=BASE_CONFIG["context_length"],
top_k=50,
temperature=1.5,
eos_id=semicolon_id
)
print("Output text:\n", token_ids_to_text(token_ids, tokenizer))

Output text:
 Every effort moves you as far as the natural capacity is capable," the lawyer wrote, "which permits extraordinary actions." "That includes (dressing


In [95]:
class WikiSQLDataset(Dataset):
    def __init__(self, file_path, tokenizer, max_length=256):
        self.samples = []
        self.tokenizer = tokenizer
        self.max_length = max_length

        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                question, sql = line.strip().split(",", 1)

                text = (
                    f"Question: {question}\n"
                    f"SQL: {sql}"
                )

                self.samples.append(text)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):

        text = self.samples[idx]

        token_ids = self.tokenizer.encode(text)

        token_ids = token_ids[:self.max_length]

        if len(token_ids) < self.max_length:
            token_ids += [50256] * (self.max_length - len(token_ids))

        input_ids = torch.tensor(token_ids)

        attention_mask = (input_ids != 50256).long()

        labels = input_ids.clone()

        labels[input_ids == 50256] = -100

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels
        }

In [96]:
train_dataset = WikiSQLDataset(
    "dataset/train.csv",
    tokenizer,
    max_length=256
)

validation_dataset = WikiSQLDataset(
    "dataset/validation.csv",
    tokenizer,
    max_length=256
)

test_dataset = WikiSQLDataset(
    "dataset/test.csv",
    tokenizer,
    max_length=256
)

In [97]:
num_workers = 24
batch_size = 24

train_loader = DataLoader(
    train_dataset,
    batch_size,
    shuffle=True,
    num_workers=num_workers,
    drop_last=True
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size,
    shuffle=False,
    num_workers=num_workers,
    drop_last=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size,
    shuffle=False,
    num_workers=num_workers,
    drop_last=False
)

In [98]:
for param in model.parameters():
    param.requires_grad = False

for param in model.trf_blocks[-1].parameters():
    param.requires_grad = True
for param in model.final_norm.parameters():
    param.requires_grad = True

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=0.01
)

criterion = torch.nn.CrossEntropyLoss(ignore_index=-100)

train_model(
    model,
    train_loader,
    validation_loader,
    optimizer,
    criterion,
    device,
    num_epochs=2
)

In [112]:
prompt = """What is the location when the status is in service as coaching stock?
SQL:"""

output = evaluate_generation(
    model=model,
    tokenizer=tokenizer,
    prompt=prompt,
    device=device,
    max_new_tokens=50
)

print(output)

What is the location when the status is in service as coaching stock?
SQL: SELECT Location FROM table WHERE Status = coaching stock


In [107]:
torch.save(model.state_dict(), "models/modelo_03.pth")